In [1]:
import numpy as np
import pandas as pd
import os
import glob
import itertools
import scanpy as sc
import natsort
import json

from scroutines import basicu

import seaborn as sns

In [2]:
# ddir = '/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/data/cheng21_cell_scrna/organized'
# outdir = '/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results/GABAARs'
# !mkdir -p $outdir

In [3]:
def get_cond_from_biosample(sample):
    """
    """
    cond = sample[:-1]
    if not cond.endswith('DR'): 
        cond = cond+'NR'
        
    return cond

In [4]:
%%time
f = '/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/data/v1_multiome/superdupermegaRNA_hasraw.h5ad'
adata_all = sc.read(f)

CPU times: user 6.13 s, sys: 25.7 s, total: 31.8 s
Wall time: 2min 36s


In [5]:
adata_all = adata_all[adata_all.obs['Study']=='2023 Multiome']
adata_all = adata_all[adata_all.obs['Subclass']=='Astro']
adata_all

View of AnnData object with n_obs × n_vars = 14028 × 16572
    obs: 'Age', 'Doublet', 'Doublet Score', 'n_counts', 'n_genes', 'percent_mito', 'sample', 'Type', 'Subclass', 'Class', 'Sample', 'total_counts', 'pct_counts_mt', 'n_genes_by_counts', 'total_counts_mt', 'Doublet?', 'Study', 'Type_leiden'
    var: 'feature_types'

In [6]:
adata_all.X = adata_all.raw.X

In [7]:
adata_all.X.data

array([2., 2., 5., ..., 2., 1., 3.], dtype=float32)

In [8]:
print(adata_all.obs['Sample'].unique().astype(str))

['P6b' 'P6c' 'P6a' 'P8a' 'P8b' 'P8c' 'P10a' 'P10b' 'P12a' 'P12c' 'P12b'
 'P12DRb' 'P12DRa' 'P14b' 'P14a' 'P14DRb' 'P14DRa' 'P17a' 'P17b' 'P17DRa'
 'P17DRb' 'P21b' 'P21a' 'P21DRb' 'P21DRa']


In [9]:
adata_all.obs['cond'] = adata_all.obs['Sample'].apply(lambda x: get_cond_from_biosample(x))
adata_all.obs['biosample'] = adata_all.obs['Sample']

/tmp/ipykernel_738/2933899518.py:1: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata_all.obs['cond'] = adata_all.obs['Sample'].apply(lambda x: get_cond_from_biosample(x))


In [10]:
unq_samples = natsort.natsorted(adata_all.obs['Sample'].unique())
print(len(unq_samples))
print(unq_samples) 

unq_conds   = natsort.natsorted(adata_all.obs['cond'].unique())
nf = len(unq_conds)
print(nf)
print(unq_conds)

25
['P6a', 'P6b', 'P6c', 'P8a', 'P8b', 'P8c', 'P10a', 'P10b', 'P12DRa', 'P12DRb', 'P12a', 'P12b', 'P12c', 'P14DRa', 'P14DRb', 'P14a', 'P14b', 'P17DRa', 'P17DRb', 'P17a', 'P17b', 'P21DRa', 'P21DRb', 'P21a', 'P21b']
11
['P6NR', 'P8NR', 'P10NR', 'P12DR', 'P12NR', 'P14DR', 'P14NR', 'P17DR', 'P17NR', 'P21DR', 'P21NR']


In [ ]:
outdir = '/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/data/v1_dev_merged'
fout = os.path.join(outdir, 'yoo25_astro.h5ad')
adata_all.write(fout)